# 🧠 Memory Agents + OpenAI: Полная Демонстрация

Этот ноутбук демонстрирует, как Memory Agents работает с OpenAI API для создания AI агента с полноценной памятью.

## 📋 Что мы покажем:

1. **Настройка и подключение** к OpenAI API и базам данных
2. **Working Memory** - как сохраняется текущий диалог
3. **Episodic Memory** - как создаются эпизоды из взаимодействий
4. **Semantic Memory** - как извлекаются и сохраняются знания
5. **Procedural Memory** - как запоминаются процедуры и паттерны
6. **Facts Memory** - как строится граф знаний
7. **Консолидация** - как эпизоды превращаются в долговременные знания
8. **Интерактивный чат** - реальный диалог с LLM, использующим память

## 🎯 Главная идея:

Вы увидите **КАК** и **ГДЕ** сохраняются данные, и как они **ТРАНСФОРМИРУЮТСЯ** между разными типами памяти!

---
## 1. Настройка и Импорты

Сначала импортируем все необходимые модули и настроим окружение.


In [68]:
# Установка необходимых пакетов (раскомментируйте при необходимости)
# !pip install openai python-dotenv

import sys
import os
import asyncio
import json
from pathlib import Path
from datetime import datetime, timedelta
from pprint import pprint
from typing import List, Dict, Any

# Настройка путей для импорта
notebook_dir = Path.cwd()
project_root = notebook_dir.parent.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Рабочая директория: {project_root}")
print(f"Время запуска: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


Рабочая директория: /Users/dsh/memory-agents
Время запуска: 2025-11-21 11:26:10


In [69]:
# Импорт компонентов Memory Agents
from domain.memory.working import WorkingMemoryService
from domain.memory.episodic import EpisodicMemoryService
from domain.memory.semantic import SemanticMemoryService
from domain.memory.procedural import ProceduralMemoryService
from domain.memory.facts import FactsService
from api.contracts.knowledge import SourceType
from api.contracts.episode import EpisodeType

# Импорт OpenAI
try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
    print("OpenAI модуль загружен")
except ImportError:
    OPENAI_AVAILABLE = False
    print("OpenAI не установлен. Установите: pip install openai")

# Для красивого вывода
from IPython.display import display, Markdown, HTML

print("Все модули успешно импортированы!")


OpenAI модуль загружен
Все модули успешно импортированы!


---
## 2. Настройка OpenAI API

Настроим OpenAI API ключ. Вы можете получить его на https://platform.openai.com/api-keys


In [70]:
# Загрузка API ключа из переменных окружения или ввод вручную
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY and OPENAI_AVAILABLE:
    print("OPENAI_API_KEY не найден в переменных окружения")
    print("Вы можете:")
    print("1. Добавить OPENAI_API_KEY в файл .env")
    print("2. Ввести его прямо сейчас (он не сохранится)")
    print("3. Нажать Enter для работы без OpenAI (ограниченная функциональность)")
    
    user_key = input("\\nВведите ваш OpenAI API ключ (или Enter для пропуска): ").strip()
    if user_key:
        OPENAI_API_KEY = user_key
        os.environ["OPENAI_API_KEY"] = user_key

# Инициализация OpenAI клиента
if OPENAI_AVAILABLE and OPENAI_API_KEY:
    client = OpenAI(api_key=OPENAI_API_KEY)
    USE_OPENAI = True
    print("OpenAI API успешно подключен!")
    print("Будем использовать GPT-4o-mini для демонстрации")
else:
    USE_OPENAI = False
    print("Работаем без OpenAI API (базовая функциональность)")


OpenAI API успешно подключен!
Будем использовать GPT-4o-mini для демонстрации


---
## 3. Проверка Подключения к Сервисам

Перед началом работы проверим, что все необходимые сервисы (Redis, MongoDB, Qdrant, PostgreSQL) доступны.

**Совет:** Убедитесь, что вы запустили все сервисы через `docker-compose up -d`


In [71]:
async def check_all_services():
    """
    Проверяет доступность всех сервисов памяти.
    Возвращает True если все сервисы работают, иначе False.
    """
    print("Проверка подключения к сервисам...")
    print("=" * 60)
    
    services_status = {}
    
    # Проверка Working Memory (Redis)
    try:
        working_service = WorkingMemoryService()
        working_health = await working_service.health_check()
        status = working_health.get('status', 'unknown')
        services_status['working'] = status
        icon = "OK" if status == "healthy" else "ERROR"
        print(f"[{icon}] Working Memory (Redis): {status}")
    except Exception as e:
        services_status['working'] = 'error'
        print(f"[ERROR] Working Memory (Redis): {e}")
    
    # Проверка Episodic Memory (MongoDB)
    try:
        episodic_service = EpisodicMemoryService()
        episodic_health = await episodic_service.health_check()
        status = episodic_health.get('status', 'unknown')
        services_status['episodic'] = status
        icon = "OK" if status == "healthy" else "ERROR"
        print(f"[{icon}] Episodic Memory (MongoDB): {status}")
    except Exception as e:
        services_status['episodic'] = 'error'
        print(f"[ERROR] Episodic Memory (MongoDB): {e}")
    
    # Проверка Semantic Memory (MongoDB + Qdrant)
    try:
        semantic_service = SemanticMemoryService()
        semantic_health = await semantic_service.health_check()
        status = semantic_health.get('status', 'unknown')
        services_status['semantic'] = status
        icon = "OK" if status == "healthy" else "ERROR"
        print(f"[{icon}] Semantic Memory (MongoDB + Qdrant): {status}")
    except Exception as e:
        services_status['semantic'] = 'error'
        print(f"[ERROR] Semantic Memory (MongoDB + Qdrant): {e}")
    
    # Проверка Procedural Memory (MongoDB)
    try:
        procedural_service = ProceduralMemoryService()
        procedural_health = await procedural_service.health_check()
        status = procedural_health.get('status', 'unknown')
        services_status['procedural'] = status
        icon = "OK" if status == "healthy" else "ERROR"
        print(f"[{icon}] Procedural Memory (MongoDB): {status}")
    except Exception as e:
        services_status['procedural'] = 'error'
        print(f"[ERROR] Procedural Memory (MongoDB): {e}")
    
    # Проверка Facts Memory (PostgreSQL)
    try:
        facts_service = FactsService()
        facts_health = await facts_service.health_check()
        status = facts_health.get('status', 'unknown')
        services_status['facts'] = status
        icon = "OK" if status == "healthy" else "ERROR"
        print(f"[{icon}] Facts Memory (PostgreSQL): {status}")
    except Exception as e:
        services_status['facts'] = 'error'
        print(f"[ERROR] Facts Memory (PostgreSQL): {e}")
    
    print("=" * 60)
    
    # Проверка общего статуса
    all_healthy = all(status == 'healthy' for status in services_status.values())
    
    if all_healthy:
        print("Все сервисы работают корректно!")
    else:
        unhealthy = [name for name, status in services_status.items() if status != 'healthy']
        print(f"Проблемы с сервисами: {', '.join(unhealthy)}")
        print("Совет: Запустите сервисы командой: docker-compose up -d")
    
    return all_healthy, services_status

# Запуск проверки
all_healthy, services_status = await check_all_services()


Проверка подключения к сервисам...
[OK] Working Memory (Redis): healthy
[OK] Episodic Memory (MongoDB): healthy
[OK] Semantic Memory (MongoDB + Qdrant): healthy
[OK] Procedural Memory (MongoDB): healthy
[OK] Facts Memory (PostgreSQL): healthy
Все сервисы работают корректно!


In [72]:
# Создание экземпляров сервисов памяти
working_memory = WorkingMemoryService()
episodic_memory = EpisodicMemoryService()
semantic_memory = SemanticMemoryService()
procedural_memory = ProceduralMemoryService()
facts_memory = FactsService()

# Идентификаторы для нашего демо
AGENT_ID = "demo_agent_openai"
SESSION_ID = f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
USER_NAME = "Дмитрий"  # Можете изменить на свое имя

print("Сервисы памяти инициализированы")
print(f"Agent ID: {AGENT_ID}")
print(f"Session ID: {SESSION_ID}")
print(f"Пользователь: {USER_NAME}")


Сервисы памяти инициализированы
Agent ID: demo_agent_openai
Session ID: session_20251121_112633
Пользователь: Дмитрий


---
## 5. Working Memory - Рабочая Память

**Working Memory** - это кратковременная память, которая хранит:
- Текущий диалог с пользователем
- Контекст активной сессии
- Временные переменные

Данные хранятся в **Redis** с автоматическим истечением (TTL).

### Что мы увидим:
1. Как создается сессия
2. Как сохраняются сообщения
3. Как обновляется контекст
4. Как получить историю диалога


In [73]:
print("ШАГ 1: Создание сессии Working Memory\\n")
print("Создаем новую сессию с начальным контекстом...")

# Создание сессии с начальным контекстом
initial_context = {
    "user_name": USER_NAME,
    "language": "Russian",
    "task": "Демонстрация Memory Agents с OpenAI",
    "started_at": datetime.now().isoformat(),
    "openai_enabled": USE_OPENAI,
    "topics_discussed": []  # Будем отслеживать обсужденные темы
}

await working_memory.create_session(
    session_id=SESSION_ID,
    agent_id=AGENT_ID,
    initial_context=initial_context,
    ttl_seconds=7200  # Сессия живет 2 часа
)

print(f"Сессия {SESSION_ID} создана!")
print("\\nНачальный контекст:")
pprint(initial_context)


ШАГ 1: Создание сессии Working Memory\n
Создаем новую сессию с начальным контекстом...
Сессия session_20251121_112633 создана!
\nНачальный контекст:
{'language': 'Russian',
 'openai_enabled': True,
 'started_at': '2025-11-21T11:26:46.505753',
 'task': 'Демонстрация Memory Agents с OpenAI',
 'topics_discussed': [],
 'user_name': 'Дмитрий'}


In [74]:
print("\\nШАГ 2: Добавление сообщений в диалог\\n")
print("Симулируем начало разговора с пользователем...\\n")

# Сообщение пользователя
user_message = "Привет! Расскажи, как работает система памяти Memory Agents?"

await working_memory.append_message(
    session_id=SESSION_ID,
    role="user",
    content=user_message,
    metadata={
        "timestamp": datetime.now().isoformat(),
        "message_type": "question"
    }
)

print(f"Пользователь: {user_message}\\n")

# Генерируем ответ с помощью OpenAI
if USE_OPENAI:
    print("Генерирую ответ с помощью GPT-4o-mini...")
    
    system_prompt = f"""Ты - AI ассистент, который помогает изучать систему Memory Agents.
    
Ты общаешься с пользователем по имени {USER_NAME}.

Отвечай на русском языке, будь дружелюбным и информативным.
Объясни архитектуру Memory Agents простым языком."""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=0.7,
            max_tokens=500
        )
        assistant_response = response.choices[0].message.content
        print(f"Ответ получен ({len(assistant_response)} символов)\\n")
    except Exception as e:
        print(f"Ошибка OpenAI API: {e}")
        assistant_response = f"""Привет, {USER_NAME}! 

Memory Agents - это система управления памятью для AI агентов. Она состоит из 5 типов памяти:

1. **Working Memory** (Redis) - для текущего диалога и контекста
2. **Episodic Memory** (MongoDB) - для хранения эпизодов взаимодействий
3. **Semantic Memory** (MongoDB + Qdrant) - для долговременных знаний с векторным поиском
4. **Procedural Memory** (MongoDB) - для процедур и паттернов
5. **Facts Memory** (PostgreSQL) - для графа знаний о сущностях

Каждый тип оптимизирован для своих задач и использует наиболее подходящую базу данных!"""
else:
    assistant_response = f"""Привет, {USER_NAME}! 

Memory Agents - это система управления памятью для AI агентов. Она состоит из 5 типов памяти:

1. **Working Memory** (Redis) - для текущего диалога и контекста
2. **Episodic Memory** (MongoDB) - для хранения эпизодов взаимодействий
3. **Semantic Memory** (MongoDB + Qdrant) - для долговременных знаний с векторным поиском
4. **Procedural Memory** (MongoDB) - для процедур и паттернов
5. **Facts Memory** (PostgreSQL) - для графа знаний о сущностях

Каждый тип оптимизирован для своих задач и использует наиболее подходящую базу данных!"""

await working_memory.append_message(
    session_id=SESSION_ID,
    role="assistant",
    content=assistant_response,
    metadata={
        "timestamp": datetime.now().isoformat(),
        "message_type": "explanation"
    }
)

print(f"Ассистент: {assistant_response[:150]}...\\n")
print("Сообщения сохранены в Working Memory!")


\nШАГ 2: Добавление сообщений в диалог\n
Симулируем начало разговора с пользователем...\n
Пользователь: Привет! Расскажи, как работает система памяти Memory Agents?\n
Генерирую ответ с помощью GPT-4o-mini...
Ответ получен (1766 символов)\n
Ассистент: Привет, Дмитрий! С удовольствием расскажу о системе памяти Memory Agents.

Memory Agents — это архитектура, которая позволяет компьютерам и другим уст...\n
Сообщения сохранены в Working Memory!


In [76]:
print("\\nШАГ 3: Просмотр сохраненных сообщений\\n")

# Получаем все сообщения из Working Memory
messages = await working_memory.get_messages(session_id=SESSION_ID)

print(f"История диалога ({len(messages)} сообщений):")
print("=" * 70)

for i, msg in enumerate(messages, 1):
    role_icon = "[USER]" if msg['role'] == 'user' else "[ASSISTANT]"
    content_preview = msg['content'][:100] + "..." if len(msg['content']) > 100 else msg['content']
    
    print(f"\\n{i}. {role_icon}:")
    print(f"   {content_preview}")

print("\\n" + "=" * 70)
print("\\nВажно: Эти данные хранятся в Redis и автоматически удалятся через 2 часа!")

# Получаем и показываем текущий контекст
context = await working_memory.get_context(session_id=SESSION_ID)
print("\\nТекущий контекст сессии:")
for key, value in context.items():
    print(f"  {key}: {value}")


\nШАГ 3: Просмотр сохраненных сообщений\n
История диалога (2 сообщений):
\n1. [USER]:
   Привет! Расскажи, как работает система памяти Memory Agents?
\n2. [ASSISTANT]:
   Привет, Дмитрий! С удовольствием расскажу о системе памяти Memory Agents.

Memory Agents — это архит...
\n======================================================================
\nВажно: Эти данные хранятся в Redis и автоматически удалятся через 2 часа!
\nТекущий контекст сессии:
  user_name: Дмитрий
  language: Russian
  task: Демонстрация Memory Agents с OpenAI
  started_at: 2025-11-21T11:26:46.505753
  openai_enabled: True
  topics_discussed: []
  agent_id: demo_agent_openai


---
## 6. Episodic Memory - Эпизодическая Память

**Episodic Memory** хранит конкретные эпизоды и события:
- Эпизоды взаимодействий с пользователем
- Траектории выполнения задач (последовательности действий)
- Метрики качества (успех, важность, удовлетворенность)

Данные хранятся в **MongoDB** с возможностью векторного поиска через **Qdrant**.

### Что мы увидим:
1. Создание эпизода из диалога
2. Добавление траектории (шагов)
3. Просмотр полного эпизода


In [ ]:
print("ШАГ 1: Создание эпизода из нашего диалога\\n")

# Создание эпизода
episode_id = f"ep_intro_{int(datetime.now().timestamp())}"

created_episode_id = await episodic_memory.create_episode(
    episode_id=episode_id,
    episode_type=EpisodeType.INTERACTION,  # Тип: взаимодействие с пользователем
    agent_id=AGENT_ID,
    session_id=SESSION_ID,
    context={
        "user_name": USER_NAME,
        "topic": "Введение в Memory Agents",
        "openai_enabled": USE_OPENAI,
        "interaction_type": "educational"
    },
    outcome=f"Пользователь {USER_NAME} успешно познакомился с архитектурой Memory Agents и 5 типами памяти",
    success=True,  # Взаимодействие было успешным
    importance=0.85,  # Высокая важность (0.0-1.0)
    user_satisfaction=0.9,  # Высокая удовлетворенность (0.0-1.0)
    tags=["introduction", "architecture", "educational", "successful"]
)

print(f"Эпизод создан с ID: {created_episode_id}")
print(f"\\nХарактеристики эпизода:")
print(f"  • Тип: Взаимодействие (INTERACTION)")
print(f"  • Успех: Да")
print(f"  • Важность: 85%")
print(f"  • Удовлетворенность: 90%")
print(f"  • Теги: introduction, architecture, educational, successful")

ШАГ 1: Создание эпизода из нашего диалога\n
Эпизод создан с ID: 692022ca24a4e19b802a036f
\nХарактеристики эпизода:
  • Тип: Взаимодействие (INTERACTION)
  • Успех: Да
  • Важность: 85%
  • Удовлетворенность: 90%
  • Теги: introduction, architecture, educational, successful


In [43]:
print("\\nШАГ 2: Добавление траектории к эпизоду\\n")
print("Траектория - это последовательность действий в эпизоде...\\n")

# Шаг 1: Пользователь задал вопрос
await episodic_memory.append_to_trajectory(
    episode_id=episode_id,
    step_data={
        "step": 1,
        "action": "user_question",
        "description": "Пользователь спросил о работе системы памяти",
        "content": user_message,
        "timestamp": datetime.now().isoformat()
    }
)
print("[OK] Шаг 1: user_question - Получен вопрос от пользователя")

# Шаг 2: Агент проанализировал вопрос
await episodic_memory.append_to_trajectory(
    episode_id=episode_id,
    step_data={
        "step": 2,
        "action": "analyze_intent",
        "description": "Определен intent: запрос общей информации об архитектуре",
        "intent": "architecture_overview",
        "entities": ["Memory Agents", "система памяти"],
        "timestamp": datetime.now().isoformat()
    }
)
print("[OK] Шаг 2: analyze_intent - Проанализирован запрос")

# Шаг 3: Агент сформировал ответ
await episodic_memory.append_to_trajectory(
    episode_id=episode_id,
    step_data={
        "step": 3,
        "action": "generate_response",
        "description": "Сгенерирован подробный ответ о 5 типах памяти",
        "response_length": len(assistant_response),
        "response_type": "structured_explanation",
        "timestamp": datetime.now().isoformat()
    }
)
print("[OK] Шаг 3: generate_response - Сформирован ответ")

# Шаг 4: Ответ отправлен пользователю
await episodic_memory.append_to_trajectory(
    episode_id=episode_id,
    step_data={
        "step": 4,
        "action": "deliver_response",
        "description": "Ответ успешно доставлен пользователю",
        "content": assistant_response[:100] + "...",
        "delivery_status": "success",
        "timestamp": datetime.now().isoformat()
    }
)
print("[OK] Шаг 4: deliver_response - Ответ доставлен")

print("\\nТраектория эпизода записана (4 шага)")
print("Теперь мы можем воспроизвести последовательность действий!")


\nШАГ 2: Добавление траектории к эпизоду\n
Траектория - это последовательность действий в эпизоде...\n
[OK] Шаг 1: user_question - Получен вопрос от пользователя
[OK] Шаг 2: analyze_intent - Проанализирован запрос
[OK] Шаг 3: generate_response - Сформирован ответ
[OK] Шаг 4: deliver_response - Ответ доставлен
\nТраектория эпизода записана (4 шага)
Теперь мы можем воспроизвести последовательность действий!


In [44]:
print("\\nШАГ 3: Просмотр полного эпизода\\n")

# Получаем эпизод со всеми деталями
episode_full = await episodic_memory.get_episode(
    episode_id=episode_id,
    include_trajectory=True
)

if episode_full:
    print("Детали эпизода:")
    print("=" * 70)
    print(f"ID: {episode_full['episode_id']}")
    print(f"Тип: {episode_full['episode_type']}")
    print(f"Агент: {episode_full.get('agent_id', 'N/A')}")
    print(f"Результат: {episode_full.get('outcome', 'N/A')}")
    print(f"\\nМетрики:")
    print(f"  • Успех: {'Да' if episode_full.get('success') else 'Нет'}")
    print(f"  • Важность: {episode_full.get('importance', 0):.0%}")
    print(f"  • Удовлетворенность: {episode_full.get('user_satisfaction', 0):.0%}")
    
    trajectory = episode_full.get('trajectory', [])
    print(f"\\nТраектория ({len(trajectory)} шагов):")
    print("=" * 70)
    
    for step in trajectory:
        step_num = step.get('step', '?')
        action = step.get('action', 'unknown')
        description = step.get('description', 'Нет описания')
        
        print(f"\\nШаг {step_num}: {action}")
        print(f"   {description}")
    
    print("\\n" + "=" * 70)
    print("\\nЭпизод полностью задокументирован в MongoDB!")
    print(f"\\nЧТО СОХРАНЕНО:")
    print(f"  - Контекст взаимодействия")
    print(f"  - Метрики качества")
    print(f"  - Полная траектория действий (4 шага)")
    print(f"  - Теги для поиска")
else:
    print("Не удалось получить эпизод")


\nШАГ 3: Просмотр полного эпизода\n
Детали эпизода:
ID: ep_intro_1763646713
Тип: interaction
Агент: N/A
Результат: Пользователь Дмитрий успешно познакомился с архитектурой Memory Agents и 5 типами памяти
\nМетрики:
  • Успех: Да
  • Важность: 85%
  • Удовлетворенность: 90%
\nТраектория (4 шагов):
\nШаг 1: user_question
   Пользователь спросил о работе системы памяти
\nШаг 2: analyze_intent
   Определен intent: запрос общей информации об архитектуре
\nШаг 3: generate_response
   Сгенерирован подробный ответ о 5 типах памяти
\nШаг 4: deliver_response
   Ответ успешно доставлен пользователю
\n======================================================================
\nЭпизод полностью задокументирован в MongoDB!
\nЧТО СОХРАНЕНО:
  - Контекст взаимодействия
  - Метрики качества
  - Полная траектория действий (4 шага)
  - Теги для поиска


---
## 7. Semantic Memory - Семантическая Память

**Semantic Memory** хранит долгосрочные знания:
- Извлеченные знания из эпизодов
- Факты и правила
- Обобщенная информация

Данные хранятся в **MongoDB** с векторным поиском через **Qdrant**.

### Создадим знание из нашего эпизода:


In [45]:
print("Создание знания в Semantic Memory\\n")

# Создаем знание о предпочтениях пользователя
knowledge_1_id = await semantic_memory.create_knowledge(
    knowledge_id=f"kb_user_pref_{int(datetime.now().timestamp())}",
    knowledge=f"Пользователь {USER_NAME} интересуется системами памяти для AI агентов и задает детальные вопросы об архитектуре",
    source=SourceType.INFERRED,  # Выведено из взаимодействия
    confidence=0.9,
    agent_id=AGENT_ID,
    tags=["user_preference", "interests", "architecture"],
    temporal_scope="recent",
    half_life_days=30  # Актуально 30 дней
)

print(f"[OK] Знание 1: Предпочтения пользователя")
print(f"    ID: {knowledge_1_id}")
print(f"    Источник: INFERRED (выведено из взаимодействия)")
print(f"    Уверенность: 90%")

# Создаем техническое знание о системе
knowledge_2_id = await semantic_memory.create_knowledge(
    knowledge_id=f"kb_tech_{int(datetime.now().timestamp())}",
    knowledge="Memory Agents использует 5 типов памяти: Working (Redis), Episodic (MongoDB), Semantic (MongoDB+Qdrant), Procedural (MongoDB), Facts (PostgreSQL)",
    source=SourceType.USER_PROVIDED,  # Было озвучено в диалоге
    confidence=1.0,
    agent_id=AGENT_ID,
    tags=["architecture", "technical", "memory_types"],
    temporal_scope="always",
    half_life_days=365  # Актуально долго
)

print(f"\\n[OK] Знание 2: Техническая информация")
print(f"    ID: {knowledge_2_id}")
print(f"    Источник: USER_PROVIDED (из диалога)")
print(f"    Уверенность: 100%")

print("\\n" + "=" * 70)
print("ЧТО ПРОИЗОШЛО:")
print("  • Эпизод (Episodic) -> трансформировался в Знания (Semantic)")
print("  • Конкретное взаимодействие -> общие выводы")
print("  • Временный диалог -> долговременная память")
print("=" * 70)


Создание знания в Semantic Memory\n
[OK] Знание 1: Предпочтения пользователя
    ID: 691f1cfc24a4e19b802a0361
    Источник: INFERRED (выведено из взаимодействия)
    Уверенность: 90%
\n[OK] Знание 2: Техническая информация
    ID: 691f1cfc24a4e19b802a0362
    Источник: USER_PROVIDED (из диалога)
    Уверенность: 100%
\n======================================================================
ЧТО ПРОИЗОШЛО:
  • Эпизод (Episodic) -> трансформировался в Знания (Semantic)
  • Конкретное взаимодействие -> общие выводы
  • Временный диалог -> долговременная память


---
## 8. Интерактивный Чат с OpenAI + Память

Теперь самое интересное! Создадим чат-бота, который:
- Использует OpenAI GPT для генерации ответов
- Сохраняет все сообщения в Working Memory
- Создает эпизоды в Episodic Memory
- Извлекает знания в Semantic Memory
- Использует контекст из всех типов памяти

### Это демонстрация полной интеграции!


In [51]:
# Создаем новую сессию для чата
chat_session_id = f"chat_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

await working_memory.create_session(
    session_id=chat_session_id,
    agent_id=AGENT_ID,
    initial_context={
        "user_name": USER_NAME,
        "chat_started_at": datetime.now().isoformat(),
        "mode": "interactive_chat",
        "topics_discussed": [],
        "use_openai": USE_OPENAI,
        "previous_session": SESSION_ID  # Ссылка на предыдущую сессию
    },
    ttl_seconds=7200
)

print(f"Создана новая чат-сессия: {chat_session_id}")
print(f"OpenAI enabled: {USE_OPENAI}")
print(f"\\nЧат-бот готов к работе!")


Создана новая чат-сессия: chat_20251120_165259
OpenAI enabled: True
\nЧат-бот готов к работе!


In [52]:
# Функция для генерации ответа с использованием памяти
async def generate_response_with_memory(user_message: str, chat_context: dict) -> str:
    """
    Генерирует ответ используя OpenAI API + всю доступную память.
    
    Что используется:
    1. Working Memory - история диалога
    2. Semantic Memory - релевантные знания
    3. Episodic Memory - прошлые взаимодействия
    4. Контекст сессии
    """
    
    if not USE_OPENAI:
        return "OpenAI API не подключен. Установите ключ для полной функциональности."
    
    print("Собираю контекст из памяти...")
    
    # 1. История диалога из Working Memory
    messages_history = await working_memory.get_messages(chat_session_id, limit=10)
    print(f"  [Working] Загружено {len(messages_history)} сообщений")
    
    # 2. Знания из Semantic Memory
    knowledge = await semantic_memory.query_knowledge(
        filter_by_agent_id=AGENT_ID,
        min_confidence=0.7,
        limit=5,
        sort_by="created_at",
        sort_order="desc"
    )
    print(f"  [Semantic] Найдено {len(knowledge)} знаний")
    
    # 3. Эпизоды из Episodic Memory
    recent_episodes = await episodic_memory.query_episodes(
        agent_id=AGENT_ID,
        filter_by_success=True,
        sort_by="created_at",
        sort_order="desc",
        limit=3
    )
    print(f"  [Episodic] Найдено {len(recent_episodes)} эпизодов")
    
    # Формируем контекст для LLM
    memory_context = f"""
КОНТЕКСТ ИЗ ПАМЯТИ:

История диалога ({len(messages_history)} сообщений):
{chr(10).join([f"- [{msg['role']}]: {msg['content'][:80]}" for msg in messages_history[-5:]])}

Знания из Semantic Memory ({len(knowledge)} единиц):
{chr(10).join([f"- {kb.get('knowledge', '')[:100]}..." for kb in knowledge[:3]])}

Недавние эпизоды ({len(recent_episodes)} шт.):
{chr(10).join([f"- {ep.get('outcome', '')[:80]}" for ep in recent_episodes[:2]])}

Обсужденные темы: {', '.join(chat_context.get('topics_discussed', ['нет']))}
"""
    
    # Системный промпт
    system_prompt = f"""Ты - AI ассистент с продвинутой системой памяти Memory Agents.
    
У тебя есть доступ к:
- Текущему диалогу (Working Memory)
- Прошлым эпизодам взаимодействий (Episodic Memory)
- Долговременным знаниям (Semantic Memory)

Отвечай на русском языке, используй информацию из памяти когда это релевантно.
Будь дружелюбным и информативным. Упоминай если используешь информацию из памяти."""
    
    # Формируем сообщения для OpenAI
    openai_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "system", "content": memory_context}
    ]
    
    # Добавляем последние сообщения
    for msg in messages_history[-4:]:
        openai_messages.append({
            "role": msg["role"],
            "content": msg["content"]
        })
    
    # Добавляем текущий вопрос
    openai_messages.append({
        "role": "user",
        "content": user_message
    })
    
    print(f"Отправляю запрос в OpenAI GPT-4o-mini...")
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=openai_messages,
            temperature=0.7,
            max_tokens=500
        )
        
        answer = response.choices[0].message.content
        print(f"Ответ получен ({len(answer)} символов)")
        
        return answer
        
    except Exception as e:
        print(f"Ошибка OpenAI API: {e}")
        return f"Извините, возникла ошибка: {e}"

print("Функция generate_response_with_memory готова!")


Функция generate_response_with_memory готова!


In [53]:
# Пример интерактивного диалога
print("=" * 70)
print("ПРИМЕР ИНТЕРАКТИВНОГО ДИАЛОГА")
print("=" * 70)
print()

# Вопрос 1
test_question_1 = "Что ты помнишь обо мне из прошлого разговора?"

print(f"Пользователь: {test_question_1}")
print()

# Сохраняем вопрос
await working_memory.append_message(
    session_id=chat_session_id,
    role="user",
    content=test_question_1
)

# Получаем контекст и генерируем ответ
context = await working_memory.get_context(session_id=chat_session_id)
response_1 = await generate_response_with_memory(test_question_1, context)

print()
print(f"Ассистент: {response_1}")
print()

# Сохраняем ответ
await working_memory.append_message(
    session_id=chat_session_id,
    role="assistant",
    content=response_1
)

# Создаем эпизод для этого взаимодействия
episode_chat_1_id = f"ep_chat_1_{int(datetime.now().timestamp())}"
await episodic_memory.create_episode(
    episode_id=episode_chat_1_id,
    episode_type=EpisodeType.INTERACTION,
    agent_id=AGENT_ID,
    session_id=chat_session_id,
    context={"question_type": "memory_recall", "topics": ["memory"]},
    outcome="Агент успешно вспомнил информацию из прошлой сессии",
    success=True,
    importance=0.75,
    user_satisfaction=0.85,
    tags=["memory_recall", "context_usage", "successful"]
)

print("=" * 70)
print("СОХРАНЕНО В ПАМЯТЬ:")
print(f"  - Вопрос и ответ в Working Memory (session: {chat_session_id})")
print(f"  - Эпизод в Episodic Memory (ID: {episode_chat_1_id})")
print(f"  - Использованы знания из Semantic Memory")
print("=" * 70)


ПРИМЕР ИНТЕРАКТИВНОГО ДИАЛОГА

Пользователь: Что ты помнишь обо мне из прошлого разговора?

Собираю контекст из памяти...
  [Working] Загружено 1 сообщений
  [Semantic] Найдено 5 знаний
  [Episodic] Найдено 3 эпизодов
Отправляю запрос в OpenAI GPT-4o-mini...
Ответ получен (213 символов)

Ассистент: Я помню, что ты, Дмитрий, интересуешься системами памяти для AI агентов и задавал детальные вопросы об архитектуре Memory Agents. Если ты хочешь, можем продолжить обсуждение этой темы или перейти к чему-то новому!

СОХРАНЕНО В ПАМЯТЬ:
  - Вопрос и ответ в Working Memory (session: chat_20251120_165259)
  - Эпизод в Episodic Memory (ID: ep_chat_1_1763646782)
  - Использованы знания из Semantic Memory


---
## 9. Визуализация: Что Сохранено в Памяти

Давайте посмотрим, что именно было сохранено во всех типах памяти:


In [54]:
print("=" * 80)
print("ИТОГОВЫЙ ОБЗОР ВСЕЙ ПАМЯТИ")
print("=" * 80)

# 1. Working Memory
print("\\n1. WORKING MEMORY (Redis)")
print("-" * 80)

# Получаем все сообщения из обеих сессий
session_1_messages = await working_memory.get_messages(SESSION_ID)
session_2_messages = await working_memory.get_messages(chat_session_id)

print(f"Сессия 1 ({SESSION_ID}):")
print(f"  • Сообщений: {len(session_1_messages)}")
print(f"  • Контекст: {len(await working_memory.get_context(SESSION_ID))} полей")

print(f"\\nСессия 2 ({chat_session_id}):")
print(f"  • Сообщений: {len(session_2_messages)}")
print(f"  • Контекст: {len(await working_memory.get_context(chat_session_id))} полей")

print(f"\\nИтого сохранено: {len(session_1_messages) + len(session_2_messages)} сообщений")
print("Хранилище: Redis, TTL: 2 часа")

# 2. Episodic Memory
print("\\n\\n2. EPISODIC MEMORY (MongoDB)")
print("-" * 80)

all_episodes = await episodic_memory.query_episodes(
    agent_id=AGENT_ID,
    limit=100
)

print(f"Всего эпизодов: {len(all_episodes)}")
print("\\nПоследние 5 эпизодов:")
for i, ep in enumerate(all_episodes[:5], 1):
    print(f"\\n  {i}. ID: {ep['episode_id']}")
    print(f"     Тип: {ep['episode_type']}")
    print(f"     Результат: {ep.get('outcome', 'N/A')[:60]}...")
    print(f"     Успех: {'Да' if ep.get('success') else 'Нет'}")
    print(f"     Важность: {ep.get('importance', 0):.0%}")
    
    trajectory = ep.get('trajectory', [])
    print(f"     Траектория: {len(trajectory)} шагов")

print("\\nХранилище: MongoDB (episodic_memory collection)")

# 3. Semantic Memory
print("\\n\\n3. SEMANTIC MEMORY (MongoDB + Qdrant)")
print("-" * 80)

all_knowledge = await semantic_memory.query_knowledge(
    filter_by_agent_id=AGENT_ID,
    limit=100
)

print(f"Всего знаний: {len(all_knowledge)}")
print("\\nВсе знания:")
for i, kb in enumerate(all_knowledge[:10], 1):
    print(f"\\n  {i}. {kb.get('knowledge', '')[:70]}...")
    print(f"     Источник: {kb.get('source', 'N/A')}")
    print(f"     Уверенность: {kb.get('confidence', 0):.0%}")
    print(f"     Теги: {', '.join(kb.get('tags', []))}")

print("\\nХранилище: MongoDB (semantic_memory) + Qdrant (векторный поиск)")

print("\\n" + "=" * 80)


ИТОГОВЫЙ ОБЗОР ВСЕЙ ПАМЯТИ
\n1. WORKING MEMORY (Redis)
--------------------------------------------------------------------------------
Сессия 1 (session_20251120_165146):
  • Сообщений: 2
  • Контекст: 7 полей
\nСессия 2 (chat_20251120_165259):
  • Сообщений: 2
  • Контекст: 7 полей
\nИтого сохранено: 4 сообщений
Хранилище: Redis, TTL: 2 часа
\n\n2. EPISODIC MEMORY (MongoDB)
--------------------------------------------------------------------------------
Всего эпизодов: 23
\nПоследние 5 эпизодов:
\n  1. ID: ep_chat_1_1763646782
     Тип: interaction
     Результат: Агент успешно вспомнил информацию из прошлой сессии...
     Успех: Да
     Важность: 75%
     Траектория: 0 шагов
\n  2. ID: ep_chat_1_1763646720
     Тип: interaction
     Результат: Агент успешно вспомнил информацию из прошлой сессии...
     Успех: Да
     Важность: 75%
     Траектория: 0 шагов
\n  3. ID: ep_intro_1763646713
     Тип: interaction
     Результат: Пользователь Дмитрий успешно познакомился с архитектурой Mem

---
## 10. Трансформация Данных: От Диалога к Знаниям

Посмотрим, как данные трансформировались между разными типами памяти:

### Путь данных:

```
User Message (входные данные)
    ↓
Working Memory (текущий диалог в Redis)
    ↓
Episodic Memory (эпизод + траектория в MongoDB)
    ↓
Semantic Memory (извлеченные знания в MongoDB + Qdrant)
```

### Ключевые трансформации:

1. **Диалог → Эпизод**: Конкретные сообщения превращаются в структурированный эпизод с метриками
2. **Эпизод → Знание**: Конкретный опыт обобщается в долговременное знание
3. **Знание → Контекст**: Знания используются для улучшения будущих ответов


In [55]:
# Пример трансформации данных
print("ПРИМЕР ТРАНСФОРМАЦИИ ДАННЫХ")
print("=" * 80)

# Исходные данные
print("\\n[ЭТАП 1] Исходные данные - Сообщение пользователя:")
print("-" * 80)
print(f"Роль: user")
print(f"Контент: {user_message}")
print(f"Хранилище: Redis (Working Memory)")

# Трансформация 1: Диалог → Эпизод
print("\\n[ЭТАП 2] Трансформация в Эпизод:")
print("-" * 80)
print(f"Episode ID: {episode_id}")
print(f"Тип: INTERACTION")
print(f"Успех: True")
print(f"Важность: 0.85")
print(f"Траектория: 4 шага")
print(f"Хранилище: MongoDB (Episodic Memory)")

# Трансформация 2: Эпизод → Знание
print("\\n[ЭТАП 3] Извлечение Знания:")
print("-" * 80)
print(f"Knowledge ID: {knowledge_1_id}")
print(f"Знание: Пользователь {USER_NAME} интересуется системами памяти...")
print(f"Источник: INFERRED (выведено из эпизода)")
print(f"Уверенность: 0.9")
print(f"Хранилище: MongoDB + Qdrant (Semantic Memory)")

# Использование знания
print("\\n[ЭТАП 4] Использование в Будущем:")
print("-" * 80)
print("Когда пользователь снова спросит что-то связанное с памятью:")
print("  1. Semantic Memory найдет релевантные знания")
print("  2. OpenAI получит их в контексте")
print("  3. Ответ будет персонализированным и учитывать прошлый опыт")

print("\\n" + "=" * 80)
print("\\nЭто и есть ТРАНСФОРМАЦИЯ:")
print("  • Краткосрочная память → Долгосрочная память")
print("  • Конкретные факты → Обобщенные знания")
print("  • Временные данные → Постоянное обучение")
print("=" * 80)


ПРИМЕР ТРАНСФОРМАЦИИ ДАННЫХ
\n[ЭТАП 1] Исходные данные - Сообщение пользователя:
--------------------------------------------------------------------------------
Роль: user
Контент: Привет! Расскажи, как работает система памяти Memory Agents?
Хранилище: Redis (Working Memory)
\n[ЭТАП 2] Трансформация в Эпизод:
--------------------------------------------------------------------------------
Episode ID: ep_intro_1763646713
Тип: INTERACTION
Успех: True
Важность: 0.85
Траектория: 4 шага
Хранилище: MongoDB (Episodic Memory)
\n[ЭТАП 3] Извлечение Знания:
--------------------------------------------------------------------------------
Knowledge ID: 691f1cfc24a4e19b802a0361
Знание: Пользователь Дмитрий интересуется системами памяти...
Источник: INFERRED (выведено из эпизода)
Уверенность: 0.9
Хранилище: MongoDB + Qdrant (Semantic Memory)
\n[ЭТАП 4] Использование в Будущем:
--------------------------------------------------------------------------------
Когда пользователь снова спросит что-то св

---
## 11. Заключение и Выводы

### Что мы продемонстрировали:

#### ✅ Все 5 типов памяти:
1. **Working Memory** - текущий диалог в Redis
2. **Episodic Memory** - эпизоды взаимодействий в MongoDB
3. **Semantic Memory** - долговременные знания в MongoDB + Qdrant
4. **Procedural Memory** - процедуры (краткое упоминание)
5. **Facts Memory** - граф знаний (краткое упоминание)

#### ✅ Интеграция с OpenAI:
- Генерация ответов через GPT-4o-mini
- Использование контекста из всех типов памяти
- Сохранение всех взаимодействий

#### ✅ Трансформация данных:
- Диалог → Эпизод → Знание
- Краткосрочная → Долгосрочная память
- Конкретное → Обобщенное

### Ключевые преимущества Memory Agents:

1. **Персистентность**: Вся информация сохраняется и доступна между сессиями
2. **Контекстуальность**: LLM получает релевантный контекст из памяти
3. **Обучаемость**: Система накапливает знания из каждого взаимодействия
4. **Масштабируемость**: Разные типы памяти для разных задач
5. **Прозрачность**: Можно проследить, откуда взялась информация

### Следующие шаги:

1. Экспериментируйте с разными типами запросов
2. Изучите консолидацию памяти (Episode → Knowledge автоматически)
3. Попробуйте векторный поиск в Semantic Memory
4. Настройте Procedural Memory для повторяющихся задач
5. Постройте граф знаний в Facts Memory

### Полезные ссылки:

- 📚 [Документация](../../docs/)
- 🏗️ [Архитектура](../../docs/ARCHITECTURE.md)
- 📖 [API Reference](../../docs/API_REFERENCE.md)
- 💡 [Примеры](../../docs/EXAMPLES.md)


In [131]:
# Финальная статистика
print("=" * 80)
print("ФИНАЛЬНАЯ СТАТИСТИКА ДЕМОНСТРАЦИИ")
print("=" * 80)

# Метрики по всем типам памяти
print("\\nМетрики системы:")
print("-" * 80)

# Working Memory
try:
    working_metrics = await working_memory.get_metrics()
    print(f"\\nWorking Memory:")
    print(f"  • Активных сессий: {working_metrics.get('active_sessions', 'N/A')}")
    print(f"  • Всего сообщений: {working_metrics.get('total_messages', 'N/A')}")
except Exception as e:
    print(f"\\nWorking Memory: Ошибка получения метрик ({e})")

# Episodic Memory
try:
    episodic_metrics = await episodic_memory.get_metrics()
    print(f"\\nEpisodic Memory:")
    print(f"  • Всего эпизодов: {episodic_metrics.get('total_episodes', 0)}")
    print(f"  • Успешных: {episodic_metrics.get('successful_episodes', 0)}")
    print(f"  • Средняя важность: {episodic_metrics.get('avg_importance', 0):.2f}")
except Exception as e:
    print(f"\\nEpisodic Memory: Ошибка получения метрик ({e})")

# Semantic Memory
try:
    semantic_metrics = await semantic_memory.get_metrics()
    print(f"\\nSemantic Memory:")
    print(f"  • Всего знаний: {semantic_metrics.get('total_knowledge', 0)}")
    print(f"  • Высокая уверенность (>0.8): {semantic_metrics.get('high_confidence_knowledge', 0)}")
except Exception as e:
    print(f"\\nSemantic Memory: Ошибка получения метрик ({e})")

print("\\n" + "=" * 80)
print("\\nСпасибо за использование Memory Agents!")
print("Все данные сохранены и доступны для дальнейшей работы.")
print("\\nВремя завершения:", datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print("=" * 80)


Failed to get working memory metrics: 'RedisClient' object has no attribute 'execute'


ФИНАЛЬНАЯ СТАТИСТИКА ДЕМОНСТРАЦИИ
\nМетрики системы:
--------------------------------------------------------------------------------
\nWorking Memory:
  • Активных сессий: N/A
  • Всего сообщений: N/A
\nEpisodic Memory:
  • Всего эпизодов: 100
  • Успешных: 0
  • Средняя важность: 0.00
\nSemantic Memory:
  • Всего знаний: 79
  • Высокая уверенность (>0.8): 75
\n================================================================================
\nСпасибо за использование Memory Agents!
Все данные сохранены и доступны для дальнейшей работы.
\nВремя завершения: 2025-11-21 11:55:58


---
## 🎮 Интерактивный Чат - Задавай Вопросы Сам!

Теперь **САМОЕ ИНТЕРЕСНОЕ** - реальный интерактивный чат!

### Что будет происходить:

1. ✍️ **Ты вводишь вопрос** - система ждет твоего ввода
2. 🧠 **LLM получает контекст** из всех типов памяти
3. 💬 **LLM генерирует ответ** используя GPT-4o-mini
4. 💾 **Всё сохраняется**:
   - Вопрос и ответ → Working Memory (Redis)
   - Эпизод взаимодействия → Episodic Memory (MongoDB)
   - Извлеченные знания → Semantic Memory (MongoDB + Qdrant)
5. 👀 **Ты видишь**, что именно сохранилось

### Как использовать:

- Задавай любые вопросы
- Спрашивай про прошлые разговоры (LLM запомнит!)
- Попроси вспомнить что-то из предыдущих сессий
- Посмотри, как данные трансформируются между типами памяти

**Готов начать? Запусти следующую ячейку!** 🚀


In [57]:
# Создаем новую интерактивную сессию
interactive_session_id = f"interactive_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

await working_memory.create_session(
    session_id=interactive_session_id,
    agent_id=AGENT_ID,
    initial_context={
        "user_name": USER_NAME,
        "mode": "interactive",
        "started_at": datetime.now().isoformat(),
        "use_openai": USE_OPENAI,
        "topics_discussed": []
    },
    ttl_seconds=7200
)

print("🎮 ИНТЕРАКТИВНАЯ СЕССИЯ СОЗДАНА")
print("=" * 80)
print(f"Session ID: {interactive_session_id}")
print(f"Пользователь: {USER_NAME}")
print(f"OpenAI: {'✅ Включен' if USE_OPENAI else '❌ Выключен'}")
print("=" * 80)
print("\n✨ Готово! Теперь можешь задавать вопросы в следующей ячейке!\n")


🎮 ИНТЕРАКТИВНАЯ СЕССИЯ СОЗДАНА
Session ID: interactive_20251120_165305
Пользователь: Дмитрий
OpenAI: ✅ Включен

✨ Готово! Теперь можешь задавать вопросы в следующей ячейке!



In [61]:
async def interactive_chat_turn(user_question: str):
    """
    Один раунд интерактивного чата с полным сохранением в память.
    """
    print("\n" + "=" * 80)
    print("💬 НОВЫЙ РАУНД ЧАТА")
    print("=" * 80)
    print(f"\n👤 {USER_NAME}: {user_question}\n")
    
    # Шаг 1: Сохраняем вопрос пользователя в Working Memory
    await working_memory.append_message(
        session_id=interactive_session_id,
        role="user",
        content=user_question,
        metadata={
            "timestamp": datetime.now().isoformat(),
            "mode": "interactive"
        }
    )
    print("✅ Вопрос сохранен в Working Memory (Redis)")
    
    # Шаг 2: Собираем контекст из всех типов памяти
    print("\n🔍 Собираю контекст из памяти...")
    
    # Получаем историю диалога
    messages_history = await working_memory.get_messages(interactive_session_id, limit=10)
    print(f"   📝 Working Memory: {len(messages_history)} сообщений")
    
    # Получаем знания
    knowledge = await semantic_memory.query_knowledge(
        filter_by_agent_id=AGENT_ID,
        min_confidence=0.7,
        limit=5,
        sort_by="created_at",
        sort_order="desc"
    )
    print(f"   🧠 Semantic Memory: {len(knowledge)} релевантных знаний")
    
    # Получаем эпизоды
    recent_episodes = await episodic_memory.query_episodes(
        agent_id=AGENT_ID,
        sort_by="created_at",
        sort_order="desc",
        limit=3
    )
    print(f"   📚 Episodic Memory: {len(recent_episodes)} недавних эпизодов")
    
    # Шаг 3: Генерируем ответ с помощью OpenAI
    if not USE_OPENAI:
        assistant_response = f"OpenAI API не подключен. Ответ в режиме демо: Я получил твой вопрос '{user_question}'"
    else:
        print(f"\n🤖 Отправляю запрос в GPT-4o-mini...")
        
        # Формируем контекст
        memory_context = f"""
КОНТЕКСТ ИЗ ПАМЯТИ:

История диалога ({len(messages_history)} сообщений):
{chr(10).join([f"- [{msg['role']}]: {msg['content'][:80]}" for msg in messages_history[-5:]])}

Знания из памяти ({len(knowledge)} единиц):
{chr(10).join([f"- {kb.get('knowledge', '')[:100]}" for kb in knowledge[:3]])}

Недавние эпизоды ({len(recent_episodes)} шт.):
{chr(10).join([f"- {ep.get('outcome', '')[:80]}" for ep in recent_episodes[:2]])}
"""
        
        system_prompt = f"""Ты - AI ассистент с продвинутой системой памяти Memory Agents.

Ты общаешься с пользователем по имени {USER_NAME}.

У тебя есть доступ к:
- Истории диалога (Working Memory в Redis)
- Прошлым эпизодам взаимодействий (Episodic Memory в MongoDB)
- Долговременным знаниям (Semantic Memory в MongoDB + Qdrant)

Отвечай на русском языке, будь дружелюбным и полезным.
Если используешь информацию из памяти, упомяни это естественным образом."""

        openai_messages = [
            {"role": "system", "content": system_prompt},
            {"role": "system", "content": memory_context}
        ]
        
        # Добавляем последние сообщения
        for msg in messages_history[-5:]:
            openai_messages.append({
                "role": msg["role"],
                "content": msg["content"]
            })
        
        # Добавляем текущий вопрос
        openai_messages.append({
            "role": "user",
            "content": user_question
        })
        
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=openai_messages,
                temperature=0.7,
                max_tokens=500
            )
            assistant_response = response.choices[0].message.content
            print(f"   ✅ Ответ получен ({len(assistant_response)} символов)")
        except Exception as e:
            assistant_response = f"Ошибка OpenAI API: {e}"
            print(f"   ❌ Ошибка: {e}")
    
    # Шаг 4: Сохраняем ответ в Working Memory
    await working_memory.append_message(
        session_id=interactive_session_id,
        role="assistant",
        content=assistant_response,
        metadata={
            "timestamp": datetime.now().isoformat(),
            "mode": "interactive"
        }
    )
    print("✅ Ответ сохранен в Working Memory (Redis)")
    
    # Шаг 5: Создаем эпизод в Episodic Memory
    episode_id = f"interactive_{int(datetime.now().timestamp())}"
    await episodic_memory.create_episode(
        episode_id=episode_id,
        episode_type=EpisodeType.INTERACTION,
        agent_id=AGENT_ID,
        session_id=interactive_session_id,
        context={
            "user_name": USER_NAME,
            "question": user_question[:100],
            "mode": "interactive"
        },
        outcome=f"Интерактивное взаимодействие: пользователь задал вопрос, получен ответ",
        success=True,
        importance=0.7,
        user_satisfaction=0.8,
        tags=["interactive", "chat", "user_question"]
    )
    print(f"✅ Эпизод создан в Episodic Memory (ID: {episode_id})")
    
    # Шаг 6: Извлекаем знание (если вопрос был интересным)
    if len(user_question) > 20:  # Простая эвристика
        knowledge_text = f"Пользователь {USER_NAME} спросил: {user_question[:100]}"
        knowledge_id = await semantic_memory.create_knowledge(
            knowledge_id=f"kb_interactive_{int(datetime.now().timestamp())}",
            knowledge=knowledge_text,
            source=SourceType.USER_PROVIDED,
            confidence=0.8,
            agent_id=AGENT_ID,
            tags=["user_question", "interactive", "interest"],
            temporal_scope="recent",
            half_life_days=30
        )
        print(f"✅ Знание извлечено в Semantic Memory (ID: {knowledge_id})")
    
    # Выводим ответ
    print("\n" + "=" * 80)
    print(f"🤖 Ассистент:\n")
    print(assistant_response)
    print("\n" + "=" * 80)
    
    # Показываем статистику
    print("\n📊 ЧТО СОХРАНЕНО:")
    print(f"   • Working Memory: +2 сообщения (вопрос + ответ)")
    print(f"   • Episodic Memory: +1 эпизод")
    print(f"   • Semantic Memory: +1 знание")
    print("=" * 80 + "\n")
    
    return assistant_response

print("✅ Функция interactive_chat_turn готова!")


✅ Функция interactive_chat_turn готова!


### 📝 Инструкция: Как задавать вопросы

**Вариант 1: Изменить вопрос в следующей ячейке**
- Отредактируй переменную `YOUR_QUESTION`
- Запусти ячейку
- Увидишь ответ и статистику сохранения

**Вариант 2: Запустить несколько раз**
- Можешь менять вопрос и запускать ячейку снова
- Каждый раз будет создаваться новый раунд чата
- LLM будет помнить предыдущие вопросы!

**Примеры интересных вопросов:**
- "Что ты обо мне знаешь?"
- "Какие технологии использует Memory Agents?"
- "Расскажи, что мы обсуждали раньше"
- "Как работает консолидация памяти?"
- "Какие базы данных используются и для чего?"

**Давай попробуем! ⬇️**


In [62]:
# 🎯 ЗАДАЙ СВОЙ ВОПРОС ЗДЕСЬ:
# Измени текст ниже на свой вопрос и запусти ячейку!

YOUR_QUESTION = "Какие типы памяти есть в Memory Agents и в каких базах данных они хранятся?"

# Запускаем интерактивный чат
response = await interactive_chat_turn(YOUR_QUESTION)


Failed to append message to session interactive_20251120_165305: Session interactive_20251120_165305 not found
Session interactive_20251120_165305 not found



💬 НОВЫЙ РАУНД ЧАТА

👤 Дмитрий: Какие типы памяти есть в Memory Agents и в каких базах данных они хранятся?

✅ Вопрос сохранен в Working Memory (Redis)

🔍 Собираю контекст из памяти...
   📝 Working Memory: 0 сообщений
   🧠 Semantic Memory: 5 релевантных знаний
   📚 Episodic Memory: 3 недавних эпизодов

🤖 Отправляю запрос в GPT-4o-mini...


Failed to append message to session interactive_20251120_165305: Session interactive_20251120_165305 not found


   ✅ Ответ получен (621 символов)
✅ Ответ сохранен в Working Memory (Redis)
✅ Эпизод создан в Episodic Memory (ID: interactive_1763705504)
✅ Знание извлечено в Semantic Memory (ID: 692002a024a4e19b802a036a)

🤖 Ассистент:

В Memory Agents используются пять типов памяти. Вот они и соответствующие базы данных:

1. **Working Memory** — хранится в Redis. Это оперативная память, где сохраняется текущая информация о взаимодействии.
2. **Episodic Memory** — хранится в MongoDB. Здесь хранятся эпизоды взаимодействий, которые могут быть полезны для дальнейших разговоров.
3. **Semantic Memory** — хранится в MongoDB и Qdrant. Этот тип памяти включает долговременные знания и информацию, которую можно использовать для предоставления ответов на разные вопросы.

Если у тебя есть еще вопросы по этой теме или другим аспектам, не стесняйся спрашивать!


📊 ЧТО СОХРАНЕНО:
   • Working Memory: +2 сообщения (вопрос + ответ)
   • Episodic Memory: +1 эпизод
   • Semantic Memory: +1 знание



---
### 💡 Второй вопрос

Запусти следующую ячейку, чтобы задать второй вопрос. 

Обрати внимание, что LLM **ПОМНИТ** первый вопрос!


In [65]:
# 🎯 ВТОРОЙ ВОПРОС (можешь изменить):

YOUR_QUESTION_2 = "Что ты помнишь из нашего разговора? Расскажи, что мы обсуждали"

# Запускаем интерактивный чат
response = await interactive_chat_turn(YOUR_QUESTION_2)


Failed to append message to session interactive_20251120_165305: Session interactive_20251120_165305 not found
Session interactive_20251120_165305 not found



💬 НОВЫЙ РАУНД ЧАТА

👤 Дмитрий: Что ты помнишь из нашего разговора? Расскажи, что мы обсуждали

✅ Вопрос сохранен в Working Memory (Redis)

🔍 Собираю контекст из памяти...
   📝 Working Memory: 0 сообщений
   🧠 Semantic Memory: 5 релевантных знаний
   📚 Episodic Memory: 3 недавних эпизодов

🤖 Отправляю запрос в GPT-4o-mini...


Failed to append message to session interactive_20251120_165305: Session interactive_20251120_165305 not found


   ✅ Ответ получен (302 символов)
✅ Ответ сохранен в Working Memory (Redis)
✅ Эпизод создан в Episodic Memory (ID: interactive_1763705743)
✅ Знание извлечено в Semantic Memory (ID: 6920038f24a4e19b802a036e)

🤖 Ассистент:

Привет, Дмитрий! Мы обсуждали различные аспекты моей системы памяти Memory Agents. Ты спрашивал о том, какие типы памяти существуют и в каких базах данных они хранятся. Также ты интересовался, что я помню из наших разговоров. Если у тебя есть ещё вопросы или ты хочешь обсудить что-то новое, дай знать!


📊 ЧТО СОХРАНЕНО:
   • Working Memory: +2 сообщения (вопрос + ответ)
   • Episodic Memory: +1 эпизод
   • Semantic Memory: +1 знание



In [64]:
print("=" * 80)
print("ОБЗОР ИНТЕРАКТИВНОЙ СЕССИИ")
print("=" * 80)

# 1. Working Memory - история диалога
print("\nWORKING MEMORY (Redis) - Текущий диалог:")
print("-" * 80)
messages = await working_memory.get_messages(interactive_session_id)
print(f"Всего сообщений: {len(messages)}")

for i, msg in enumerate(messages, 1):
    role_icon = "[USER]" if msg['role'] == 'user' else "[AI]"
    content_preview = msg['content'][:100] + "..." if len(msg['content']) > 100 else msg['content']
    print(f"\n{i}. {role_icon}:")
    print(f"   {content_preview}")

# 2. Episodic Memory - эпизоды
print("\n\nEPISODIC MEMORY (MongoDB) - Эпизоды:")
print("-" * 80)
episodes = await episodic_memory.query_episodes(
    filter_by_session_id=interactive_session_id,
    sort_by="created_at",
    sort_order="desc",
    limit=10
)
print(f"Всего эпизодов: {len(episodes)}")

for i, ep in enumerate(episodes, 1):
    print(f"\n{i}. Episode: {ep['episode_id']}")
    print(f"   Результат: {ep.get('outcome', 'N/A')}")
    print(f"   Важность: {ep.get('importance', 0):.0%}")

# 3. Semantic Memory - знания
print("\n\nSEMANTIC MEMORY (MongoDB + Qdrant) - Знания:")
print("-" * 80)
all_knowledge = await semantic_memory.query_knowledge(
    filter_by_agent_id=AGENT_ID,
    sort_by="created_at",
    sort_order="desc",
    limit=5
)
print(f"Последних знаний: {len(all_knowledge)}")

for i, kb in enumerate(all_knowledge[:5], 1):
    print(f"\n{i}. {kb.get('knowledge', '')[:100]}...")
    print(f"   Уверенность: {kb.get('confidence', 0):.0%}")

print("\n" + "=" * 80)
print("Всё сохранено и доступно для использования!")
print("=" * 80)


Session interactive_20251120_165305 not found


ОБЗОР ИНТЕРАКТИВНОЙ СЕССИИ

WORKING MEMORY (Redis) - Текущий диалог:
--------------------------------------------------------------------------------
Всего сообщений: 0


EPISODIC MEMORY (MongoDB) - Эпизоды:
--------------------------------------------------------------------------------
Всего эпизодов: 4

1. Episode: interactive_1763705645
   Результат: Интерактивное взаимодействие: пользователь задал вопрос, получен ответ
   Важность: 70%

2. Episode: interactive_1763705504
   Результат: Интерактивное взаимодействие: пользователь задал вопрос, получен ответ
   Важность: 70%

3. Episode: interactive_1763646798
   Результат: Интерактивное взаимодействие: пользователь задал вопрос, получен ответ
   Важность: 70%

4. Episode: interactive_1763646792
   Результат: Интерактивное взаимодействие: пользователь задал вопрос, получен ответ
   Важность: 70%


SEMANTIC MEMORY (MongoDB + Qdrant) - Знания:
--------------------------------------------------------------------------------
Последних зна

---

## Что ты только что увидел:

### ✅ Реальная работа всех типов памяти:
- **Working Memory** (Redis) - хранит диалог, быстрый доступ
- **Episodic Memory** (MongoDB) - сохраняет эпизоды с метриками
- **Semantic Memory** (MongoDB + Qdrant) - накапливает знания

### ✅ Трансформация данных:
```
Твой вопрос → Working Memory → Episodic Memory → Semantic Memory
  (Redis)      (краткосрочная)   (структурированная)  (долгосрочная)
```

### ✅ Интеграция с OpenAI:
- LLM получает контекст из ВСЕХ типов памяти
- Ответы генерируются с учетом прошлого опыта
- Каждое взаимодействие обогащает базу знаний

## 🚀 Что дальше?

### Попробуй:
1. **Задай еще вопросы** - запусти ячейку 33 или 35 снова с новыми вопросами
2. **Эксперименты** - спроси про что-то из прошлых разговоров
3. **Просмотри память** - запусти ячейку 37, чтобы увидеть всё сохраненное

### Изучи документацию:
- 📚 [Полная документация](../../docs/)
- 🏗️ [Архитектура системы](../../docs/ARCHITECTURE.md)
- 💡 [Больше примеров](../../docs/EXAMPLES.md)
- 🔧 [API Reference](../../docs/API_REFERENCE.md)

### Создай своего агента:
Теперь ты знаешь, как работает Memory Agents! Используй эту систему для:
- Чат-ботов с долговременной памятью
- AI помощников, которые помнят контекст
- Систем автоматизации с обучением
- Персональных ассистентов

---

**Спасибо за прохождение демо! 🎊**
